In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date
from pyspark.sql.types import IntegerType

In [2]:
spark = SparkSession.builder \
    .appName("ProcessCustomersSilver") \
    .master("local[*]") \
    .getOrCreate()

26/02/15 20:20:16 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
df_bronze = spark.read.parquet("bronze/customers")

In [4]:
print("Bronze count:", df_bronze.count())

Bronze count: 141465


In [5]:
df_bronze.printSchema()

root
 |-- Id: string (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- RegistrationDate: string (nullable = true)
 |-- State: string (nullable = true)



In [7]:
df_bronze.show(5)

+---+---------+---------+--------------------+----------------+-----+
| Id|FirstName| LastName|               Email|RegistrationDate|State|
+---+---------+---------+--------------------+----------------+-----+
|  9|    Susan|     Levy|susan_levy@exampl...|       2022-08-2| NULL|
| 10|     Sean|     NULL|sean_green@exampl...|       2022-08-2|Texas|
| 18|    Tasha|Rodriguez|tasha_rodriguez@e...|       2022-08-2|Maine|
| 25|    Peter| Mcdowell|peter_mcdowell@ex...|       2022-08-2| NULL|
| 29|Stephanie|     NULL|stephanie_lawrenc...|       2022-08-2| Ohio|
+---+---------+---------+--------------------+----------------+-----+
only showing top 5 rows


In [9]:
df_silver = df_bronze \
    .withColumn("client_id", col("Id").cast(IntegerType())) \
    .withColumn("first_name", col("FirstName")) \
    .withColumn("last_name", col("LastName")) \
    .withColumn("email", col("Email")) \
    .withColumn("registration_date", to_date(col("RegistrationDate"))) \
    .withColumn("state", col("State")) \
    .select(
        "client_id",
        "first_name",
        "last_name",
        "email",
        "registration_date",
        "state"
    )

In [11]:
before = df_silver.count()

df_silver = df_silver.dropDuplicates(["client_id"])

after = df_silver.count()

print("Before:", before)
print("After:", after)

[Stage 8:===>                                                     (1 + 14) / 15]

Before: 141465
After: 47469


In [12]:
df_silver = df_silver.dropna(subset=["client_id"])

In [13]:
df_silver.write \
    .mode("overwrite") \
    .parquet("silver/customers")

26/02/15 20:26:09 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [14]:
df_silver \
    .repartition(1) \
    .write \
    .mode("overwrite") \
    .parquet("silver/customers")


In [15]:
df_check = spark.read.parquet("silver/customers")

df_check.show(20, truncate=False)

+---------+----------+----------+-----------------------------+-----------------+-----+
|client_id|first_name|last_name |email                        |registration_date|state|
+---------+----------+----------+-----------------------------+-----------------+-----+
|26       |NULL      |Villanueva|sarah_villanueva@example.com |2022-08-03       |Idaho|
|27       |Kimberly  |NULL      |kimberly_myers@example.com   |2022-08-01       |NULL |
|28       |Desiree   |Cain      |desiree_cain@example.com     |2022-08-05       |Maine|
|31       |Whitney   |Stark     |whitney_stark@example.com    |2022-08-05       |Idaho|
|34       |Faith     |Cabrera   |faith_cabrera@example.com    |2022-08-04       |Maine|
|44       |Matthew   |Bell      |matthew_bell@example.com     |2022-08-01       |Texas|
|53       |NULL      |Howard    |emily_howard@example.com     |2022-08-04       |Texas|
|65       |Jeremy    |Green     |jeremy_green@example.com     |2022-08-04       |Iowa |
|76       |Brandi    |Meyer     